Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/azure-arcadia/automl_arcadia_local_regression.png)

## Automated ML on Arcadia

In this notebook you will learn how to:
1. Create Azure Machine Learning Workspace object and initialize your notebook directory to easily reload this object from a configuration file.
2. Create an `Experiment` in an existing `Workspace`.
3. Configure Automated ML using `AutoMLConfig`.
4. Train the model using Arcadia.
5. Explore the results.
6. Test the best fitted model.


### Check the Azure ML Core SDK Version to Validate Your Installation

In [ ]:
import azureml.core
print("SDK Version:", azureml.core.VERSION)

## Initialize an Azure ML Workspace
### What is an Azure ML Workspace and Why Do I Need One?

An Azure ML workspace is an Azure resource that organizes and coordinates the actions of many other Azure resources to assist in executing and sharing machine learning workflows.  In particular, an Azure ML workspace coordinates storage, databases, and compute resources providing added functionality for machine learning experimentation, operationalization, and the monitoring of operationalized models.


### What do I Need?

To create or access an Azure ML workspace, you will need to import the Azure ML library and specify following information:
* A name for your workspace. You can choose one.
* Your subscription id. Use the `id` value from the `az account show` command output above.
* The resource group name. The resource group organizes Azure resources and provides a default region for the resources in the group. The resource group will be created if it doesn't exist. Resource groups can be created and viewed in the [Azure portal](https://portal.azure.com)
* Supported regions include `eastus`, `westus2`.

## Creating a Workspace
If you already have access to an Azure ML workspace you want to use, you can skip this cell.  Otherwise, this cell will create an Azure ML workspace for you in the specified subscription, provided you have the correct permissions for the given `subscription_id`.

This will fail when:
1. The workspace already exists.
2. You do not have permission to create a workspace in the resource group.
3. You are not a subscription owner or contributor and no Azure ML workspaces have ever been created in this subscription.

If workspace creation fails for any reason other than already existing, please work with your IT administrator to provide you with the appropriate permissions or to provision the required resources.

**Note:** Creation of a new workspace can take several minutes.

In [ ]:
##PUBLISHONLY
#subscription_id = "<Your SubscriptionId>" #you should be owner or contributor
#resource_group = "<Resource group - new or existing>" #you should be owner or contributor
#workspace_name = "<workspace to be created>" #your workspace name
#workspace_region = "<azureregion>" #your region

In [ ]:
##PUBLISHONLY
## Import the Workspace class and check the Azure ML SDK version.
#from azureml.core import Workspace

#ws = Workspace.create(name = workspace_name,
#                      subscription_id = subscription_id,
#                      resource_group = resource_group, 
#                      location = workspace_region,                      
#                      exist_ok=True)
#ws.get_details()

In [ ]:
# import modules
import azureml.core
import pandas as pd
from azureml.core.authentication import ServicePrincipalAuthentication
from azureml.core.workspace import Workspace
from azureml.core.experiment import Experiment
from azureml.train.automl import AutoMLConfig
from azureml.train.automl.run import AutoMLRun
import logging

import os
import random
import time
import numpy as np

# Choose a name for the experiment
experiment_name = 'automl-local-regression-Arcadia'
experiment = Experiment(ws, experiment_name)
output = {}
output['SDK version'] = azureml.core.VERSION
output['Subscription ID'] = ws.subscription_id
output['Workspace Name'] = ws.name
output['Resource Group'] = ws.resource_group
output['Location'] = ws.location
output['Experiment Name'] = experiment.name
pd.set_option('display.max_colwidth', -1)
outputDf = pd.DataFrame(data = output, index = [''])
outputDf.T


## Registering Datastore


Datastore is the way to save connection information to a storage service (e.g. Azure Blob, Azure Data Lake, Azure SQL) information to your workspace so you can access them without exposing credentials in your code. The first thing you will need to do is register a datastore, you can refer to our [python SDK documentation](https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.datastore.datastore?view=azure-ml-py) on how to register datastores. __Note: for best security practices, please do not check in code that contains registering datastores with secrets into your source control__

The code below registers a datastore pointing to a publicly readable blob container.

In [ ]:
from azureml.core import Datastore

datastore_name = 'nyctlc_demo'
container_name = 'nyctlc' 
account_name = 'automlpublicdatasets'
Datastore.register_azure_blob_container(
    workspace = ws, 
    datastore_name = datastore_name, 
    container_name = container_name, 
    account_name = account_name,
     overwrite = True
)

In [ ]:
from azureml.core import Datastore
datastore_name = 'nyctlc_demo'

from azureml.core.dataset import Dataset
from azureml.data.datapath import DataPath
from azureml.core import Datastore
from IPython.display import display
import pandas as pd

# Ingest data
datastore = Datastore.get(workspace = ws, datastore_name = datastore_name)
dataset = Dataset.Tabular.from_parquet_files((datastore, '\part*.parquet'))
# Keeping relevant features
dataset = dataset.drop_columns(columns=['startLon', 'startLat', 'endLon', 'endLat', 'rateCodeId', 'storeAndFwdFlag'])
dataset_train, dataset_test = dataset.random_split(0.9)

X_train = dataset_train.drop_columns(columns=['fareAmount'])
y_train = dataset_train.keep_columns(columns=['fareAmount'], validate=True)


## Train

Instantiate an `AutoMLConfig` object to specify the settings and data used to run the experiment.

**Note:** Set the parameter enable_onnx_compatible_models=True, if you also want to generate the ONNX compatible models. Please note, the forecasting task and TensorFlow models are not ONNX compatible yet.

|Property|Description|
|-|-|
|**task**|classification or regression|
|**primary_metric**|This is the metric that you want to optimize.|
|**iteration_timeout_minutes**|Time limit in minutes for each iteration.|
|**iterations**|Number of iterations. In each iteration AutoML trains a specific pipeline with the data.|
|**X**|(sparse) array-like, shape = [n_samples, n_features]|
|**y**|(sparse) array-like, shape = [n_samples, ], Multi-class targets.|
|**enable_onnx_compatible_models**|Enable the ONNX compatible models in the experiment.|
|**path**|Relative path to the project folder. AutoML stores configuration files for the experiment under this folder. You can specify a new empty folder.|


In [ ]:
automl_config = AutoMLConfig(task = 'regression',
                             debug_log = 'automl_errors.log',
                             primary_metric = 'normalized_root_mean_squared_error',
                             iteration_timeout_minutes = 10,
                             iterations = 20,
                             preprocess = True,
                             n_cross_validations = 2,
                             max_concurrent_iterations = 2, #spark compute size
                             verbosity = logging.INFO,
                             spark_context=sc, #spark related
                             enable_onnx_compatible_models=True, # This will generate ONNX compatible models.
                             cache_store=True,
                             X = X_train, 
                             y = y_train)

Call the `submit` method on the experiment object and pass the run configuration. Execution of local runs is synchronous. Depending on the data and the number of iterations this can run for a while.
In this example, we specify `show_output = True` to print currently running iterations to the console.

In [ ]:
local_run = experiment.submit(automl_config, show_output = True)

### Retrieve the Best ONNX Model

Below we select the best pipeline from our iterations. The `get_output` method returns the best run and the fitted model. The Model includes the pipeline and any pre-processing.  Overloads on `get_output` allow you to retrieve the best run and fitted model for *any* logged metric or for a particular *iteration*.

Set the parameter return_onnx_model=True to retrieve the best ONNX model, instead of the Python model.

In [ ]:
best_run, fitted_model = local_run.get_output(return_onnx_model=True)
print(fitted_model)

## Portal URL for Monitoring Runs


In [ ]:
more Insights of experiment
displayHTML("<a href={} target='_blank'>Your experiment in Azure Portal: {}</a>".format(local_run.get_portal_url(), local_run.id))

## Register Model


In [ ]:
#register model
description = 'AutoML Demo Model'
tags = None
model = local_run.register_model(description = description, tags = tags)
local_run.model_id


## Model Deployment and Batch Inference Using Spark


In [ ]:
from azureml.train.automl import AutoMLConfig, constants
import json
def get_onnx_res(run):
    res_path = 'onnx_resource.json'
    run.download_file(name=constants.MODEL_RESOURCE_PATH_ONNX, output_file_path=res_path)
    with open(res_path) as f:
        onnx_res = json.load(f)
    return onnx_res

In [ ]:
onnx_res = get_onnx_res(best_run)

#### Broadcast the model and onnx resource to all worker nodes for distributed inferencing using Spark.

In [ ]:
#broadcast and spark dataframe
model_broadcast = sc.broadcast(fitted_model)
onnx_res_broadcast = sc.broadcast(onnx_res)
X_test = dataset_test.drop_columns(columns=['fareAmount'])
X_df = X_test.to_pandas_dataframe()
spark_df = sqlContext.createDataFrame(X_df, X_df.columns.tolist())

#### Partitions resilient distributed data and distribute across different nodes and predict.

In [ ]:
# rdd to partition the data and call predict

def predictor(iterator):
    from azureml.core.experiment import Experiment
    from azureml.train.automl import AutoMLConfig
    from azureml.train.automl.run import AutoMLRun
    import json
    from azureml.automl.core.onnx_convert import OnnxConvertConstants
    import numpy
    import onnxruntime
    from azureml.automl.core.onnx_convert import OnnxInferenceHelper    
    
    x_array = []
    for x in iterator:
        x_array.append(x)
    x_df = pd.DataFrame(x_array)
    x_df.columns = X_df.columns
    mdl_bytes = model_broadcast.value.SerializeToString()
    print(mdl_bytes)
    onnxrt_helper = OnnxInferenceHelper(mdl_bytes, onnx_res_broadcast.value)
    x_df["Predicted_fareAmount"], pred_prob_onnx = onnxrt_helper.predict(x_df)
    return [x_df]

from pyspark import SparkContext
rdd_df = spark_df.rdd
results=rdd_df.mapPartitions(predictor).collect()
print results